In [ ]:
import deepfmkit.core as dfm
import numpy as np
from scipy.signal import welch as psd
from deepfmkit.plotting import default_rc
import matplotlib.pyplot as plt

plt.rcParams.update(default_rc)

dff = dfm.DeepFrame()

laser_config = dfm.LaserConfig(label="main_laser")
# Modulation frequency (Hz)
laser_config.fm = 1000
# Laser frequency noise at 1 Hz (Hz/rtHz)
laser_config.f_n = 1e7
# Laser amplitude noise (1/rtHz)
laser_config.amp_n = 1e-5

main_ifo_config = dfm.IfoConfig(label="dynamic_ifo")
# Reference arm length (m)
main_ifo_config.ref_arml = 0.1
# Measurement arm length (m)
main_ifo_config.meas_arml = 0.15
# Measurement arm modulation frequency (Hz)
main_ifo_config.arml_mod_f = 5.0
# Armlength modulation amplitude (m)
main_ifo_config.arml_mod_amp = 1e-8
# Armlength modulation amplitude noise (m/rtHz)
main_ifo_config.arml_n = 1e-10
# Electronic noise (V/rtHz)
main_ifo_config.s_n = 1e-5

# Target effective modulation index (rad)
m_target = 6.0
laser_config.set_df_for_effect(main_ifo_config, m_target)

main_label = "main"
main_channel = dfm.SimConfig(
    label=main_label,
    laser_config=laser_config,
    ifo_config=main_ifo_config,
    f_samp=int(200e3),  # Sampling frequency (Hz)
)
dff.sims[main_label] = main_channel

# We want a witness with this effective modulation index
m_ref = 5.99
ref_label = "reference"
dff.create_witness_channel(
    main_channel_label=main_label,
    witness_channel_label=ref_label,
    m_witness=m_ref,
    arml_n=1e-12,
    s_n=1e-3,
    s_n_alpha=2.0,
)

dff.simulate(
    label=main_label,
    witness_label=ref_label,
    n_seconds=40,
    verbose=True,
)

dff.sims[main_label].info()
dff.sims[ref_label].info()

for i, key in enumerate(dff.raws):
    print(f"Fitting channel '{key}'...")
    dff.fit(key, fit_label=f"fit_{key}")

ax = dff.plot()
plt.show()

In [ ]:
ref_phase = dff.fits["fit_"+ref_label].phi - np.mean(
    dff.fits["fit_"+ref_label].phi
)
tm_phase = dff.fits["fit_"+main_label].phi - np.mean(
    dff.fits["fit_"+main_label].phi
)
ref_m = dff.fits["fit_"+ref_label].m
tm_m = dff.fits["fit_"+main_label].m

diff_phase = tm_phase - ref_phase * m_target / m_ref

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(tm_phase, label=r"TMIFO $\Phi$")
ax.plot(ref_phase, label=r"RefIFO $\Phi$")
ax.plot(diff_phase, label=r"Diff. $\Phi$", alpha=0.5)

ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(tm_phase[0:500], label=r"TMIFO $\Phi$")
ax.plot(ref_phase[0:500], label=r"RefIFO $\Phi$")
ax.plot(diff_phase[0:500], label=r"Diff. $\Phi$", alpha=0.5)

ax.legend()
plt.show()

In [ ]:
f_fit = dff.sims[ref_label].f_fit

f2, psd2 = psd(tm_phase, fs=f_fit)
f3, psd3 = psd(ref_phase, fs=f_fit)
f4, psd4 = psd(diff_phase, fs=f_fit)

fig, ax = plt.subplots()
ax.loglog(f2, np.sqrt(psd2), label=r"TMIFO $\Phi$")
ax.loglog(f3, np.sqrt(psd3), label=r"RefIFO $\Phi$")
ax.loglog(f4, np.sqrt(psd4), label=r"Diff. $\Phi$", ls="--")

ax.legend()
plt.show()

In [ ]:
f_fit = dff.sims[ref_label].f_fit

f5, psd5 = psd(tm_m, fs=f_fit)
f6, psd6 = psd(ref_m, fs=f_fit)

fig, ax = plt.subplots()
ax.loglog(f5, np.sqrt(psd5), label=r"TMIFO $m$")
ax.loglog(f6, np.sqrt(psd6), label=r"RefIFO $m$")

ax.legend()
plt.show()